###model optimization and cross validation


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import(
train_test_split,
StratifiedKFold,
cross_val_score,
GridSearchCV
)

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import(
accuracy_score,classification_report,
confusion_matrix
)
from scipy.stats import randint

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from scipy.stats import randint

In [6]:
df = pd.read_csv("bank_marketing.csv")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,32,admin.,married,secondary,no,11434,no,no,cellular,14,jan,152,3,259,10,unknown,no
1,87,student,single,secondary,no,-1149,no,yes,cellular,9,mar,911,11,195,2,failure,no
2,62,housemaid,single,primary,no,14361,no,no,cellular,27,nov,2563,12,-1,0,unknown,yes
3,23,unemployed,single,secondary,no,28512,no,no,cellular,27,jun,696,12,-1,0,unknown,yes
4,27,housemaid,married,secondary,no,58589,yes,no,cellular,18,apr,2834,11,-1,0,unknown,yes


In [8]:
print(df.shape)
print(df.columns.to_list)

(20000, 17)
<bound method IndexOpsMixin.tolist of Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')>


In [9]:
df.columns = df.columns.str.strip()
print(df.columns.to_list)

<bound method IndexOpsMixin.tolist of Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')>


In [15]:
target = "y"
##check missing values
print(df.isnull().sum())
print(df[target].value_counts())

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64
y
yes    13307
no      6693
Name: count, dtype: int64


In [16]:
print(df[target].value_counts())

y
yes    13307
no      6693
Name: count, dtype: int64


In [20]:
df.dtypes#(check data types)

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [21]:
encoder = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = encoder.fit_transform(df[col])

C:\Users\Vipin Yadav\AppData\Local\Temp\ipykernel_22420\1602248350.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [23]:
x = df.drop(columns=[target])
y = df[target]
print("feature shape = ",x.shape)
print("target shape = ",y.shape)

feature shape =  (20000, 16)
target shape =  (20000,)


In [24]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=42)
print("training ",x_train.shape)
print("testing",x_test.shape)

training  (16000, 16)
testing (4000, 16)


In [28]:
model = DecisionTreeClassifier(random_state=42)
model.fit(x_train,y_train)
prediction = model.predict(x_test)
print("accracy ",round(accuracy_score(y_test,prediction),4))
print("classification report",(y_test,prediction))
print("confusion matrix :-\n",confusion_matrix(y_test,prediction))

accracy  0.7308
classification report (10650    0
2041     1
8668     1
1114     1
13902    0
        ..
4073     1
7442     0
9999     0
1870     0
15196    0
Name: y, Length: 4000, dtype: int64, array([0, 1, 1, ..., 0, 1, 0], shape=(4000,)))
confusion matrix :-
 [[ 841  467]
 [ 610 2082]]


In [32]:
##cross validation###
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(model,x,y,cv=cv,scoring="accuracy")
print(scores)
print("average accuracy",round(scores.mean(),4))
print("standard deviation",round(scores.std(),4))

[0.73175 0.72075 0.7305  0.7225  0.729  ]
average accuracy 0.7269
standard deviation 0.0044


In [37]:
param_grid = {
    "criterion":["gini","entropy"],
    "max_depth":[3,5,7,10,15,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,6]
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(x_train,y_train)
print("best parameters =",grid.best_params_)
print("best cv scores =",round(grid.best_score_,4))
grid_prediction = grid.predict(x_test)
print("grid search cv test accuracy=",round(accuracy_score(y_test,grid_prediction),4))

    


best parameters = {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 10}
best cv scores = 0.8262
grid search cv test accuracy= 0.8427


In [45]:
from sklearn.model_selection import RandomizedSearchCV

In [48]:
###randomized ###
param_dist = {
    "criterion":["gini","entropy"],
    "max_depth":randint(2,20),
    "min_samples_split":randint(2,15),
    "min_samples_leaf":randint(2,5)
}

random = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    random_state=42,
    scoring="accuracy",
    n_jobs=-1
)

random.fit(x_train,y_train)
print("best parameters =",random.best_params_)
print("best cv scores",round(random.best_score_),4)
random_prediction = random.predict(x_test)
print("random search accuracy",round(accuracy_score(y_test,random_prediction),4))


best parameters = {'criterion': 'gini', 'max_depth': 8, 'min_samples_leaf': 3, 'min_samples_split': 4}
best cv scores 1 4
random search accuracy 0.8413
